## Modelos de Machine Learning: XGBoost / LightGBM — Sistema de Predicción Energética en Barcelona

> **Contexto:** A partir de `dataset_features` (notebooks 03 y 04), y tras establecer los baselines
> SARIMA/SARIMAX (notebook 05), esta fase entrena los modelos de **machine learning** del TFM. El
> objetivo es comprobar si los modelos de árboles —capaces de capturar relaciones no lineales e
> interacciones— **superan a los baselines lineales** bajo el **mismo arnés de evaluación**.

### Principio de comparación justa

Para que la comparación con SARIMA/SARIMAX sea válida, este notebook reutiliza **exactamente**:
- el mismo **split temporal** (train 2019–2024 · validación ene–sep 2025 · test oct–nov 2025 intacto),
- las mismas **métricas** (R² primaria; MAE, RMSE, MAPE secundarias),
- los mismos **horizontes** de backtesting (24h, 48h, 72h).

> ⚠️ La diferencia clave frente a SARIMA: los árboles **no** modelan la dependencia temporal de
> forma implícita — dependen de las *features* (lags, calendario, clima) construidas en el notebook 04.

## Librerías

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pymongo import MongoClient
import warnings, time

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
# Modelos (instalar cuando toque):  pip install xgboost lightgbm
# from xgboost import XGBRegressor
# import lightgbm as lgb

warnings.filterwarnings('ignore')

plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#D3D3D3'
plt.rcParams['grid.linewidth'] = 0.4
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Paleta de color (consistente con los notebooks 03-05)
C1 = '#264653'; C2 = '#2A9D8F'; C3 = '#E9C46A'
C4 = '#F4A261'; C5 = '#E76F51'; C6 = '#A8DADC'
TITULO = '#1B3A5C'; SUBTITULO = '#C0392B'

start_time = time.time()

---
## Carga de datos

In [ ]:
client = MongoClient('mongodb://mongo:27017/')
db     = client['tfm_energy']

docs = list(db['dataset_ml'].find({}, {'_id': 0}))
df_ml   = pl.DataFrame(docs, infer_schema_length=None)

print(f"Shape: {df_ml.shape}")
print(f"Desde: {df_ml['datetime'].min()}  Hasta: {df_ml['datetime'].max()}")
print(f"Codigos postales: {df_ml['cod_postal'].n_unique()}")
print(f"Columnas ({len(df_ml.columns)}):")
print(sorted(df_ml.columns))

---
# <font color='#1B3A5C'>  **1. Configuración del Experimento** </font>

> Se fijan los **mismos cortes temporales y horizontes** que en el notebook 05 (SARIMA/SARIMAX).
> Esto es **innegociable**: solo así la comparación baseline vs. ML es justa. El conjunto de
> **test (oct–nov 2025) permanece intacto** hasta la evaluación final.

In [ ]:
# Identicos a SARIMA, SARIMAX)
INI       = '2019-01-01'             # historico completo
FIN_TRAIN = '2024-12-31 18:00:00'    # fin de train
FIN_VAL   = '2025-09-30 18:00:00'    # fin de validacion (2025-01 -> 2025-09)
# test: 2025-10-01 a  2025-11-30 (lo que queda tras FIN_VAL), INTACTO

#  4=24h, 8=48h, 12=72h
HORIZONTES = {'24h': 4, '48h': 8, '72h': 12}

TARGET = 'mwh_total'
CPS_TODOS = sorted(df_ml['cod_postal'].unique().to_list())

print(f"Train : {INI} -> {FIN_TRAIN}")
print(f"Val   : 2025-01-01 -> {FIN_VAL}")
print(f"Test  : 2025-10-01 -> {df_ml['datetime'].max()}  (intacto)")
print(f"Horizontes: {HORIZONTES}")
print(f"Target: {TARGET} | CPs: {len(CPS_TODOS)}")